<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 6: Transformers and GPT

#### Tim Moroney, 2026

A lesson where we learn about the **transformer** and the related **attention** mechanism that powers modern large language models (LLMs).  We will see how the architecture compares to our simple CLLM, and what other tricks of the trade are necessary to get a model that really generates new text.  We'll conclude by experimenting a bit with a pre-trained GPT 2 model.

# Package management

This week only we've included a pre-packaged, pre-trained GPT 2 model, so the download is bigger than usual.

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU_transformers.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1Iq_AbzI760Dw3ks3JNr7f8M1U_xQLCju`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU_transformers.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "TextEncodeBase", "ToeplitzMatrices",
           "Transformers", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentArray
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

using TextEncodeBase
using Transformers
using Transformers.HuggingFace

# Set the random seed for reproducibility
Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# No batching

In our mathematical analysis for this lesson, we return to the _pre-batch_ implementation of our model formulation.  So, the model takes only one text sequence input, not a batch of them.  This will ease the transition to the more sophisticated transformer-based approach.  Another day we (you!) can add batches back in to the picture.

## CLLM versus LLM
There is not a huge conceptual difference between the character level language model (CLLM) that we've been studying, and a large language model (LLM).  Both models are trained to predict the next token based on an input context window of tokens: $$\hat{y} = M_p(u)\qquad$$ where $$u = [u_1, u_2, \cdots, u_C]^\top$$ is the input vector of token indices.  In a CLLM, "tokens" are just single characters from a very limited vocabulary.  But in an LLM the tokens aren't just single characters any more.  They could be full words, or parts of words, and any single word could be comprised of several tokens.

Because of this, there are going to be many more tokens in the vocabulary, and a much more complex functional relationship to try to learn. As we will see, our basic architecture
$$M_p(u) = \textrm{softmax}(W_2\, \tanh.(W_1\textrm{vec}(W_e[:,u])+b_1 )+b_2)$$
that served us well until now isn't going to scale up satisfactorily to this new task.



# Tokenising

Unlike with our CLLM, there is a nontrivial first step in setting up an LLM: tokenising.  That is, turning the input text into tokens.

GPT 2 uses a [byte-pair encoding](https://en.wikipedia.org/wiki/Byte-pair_encoding) (BPE) tokeniser, the details of which are beyond the scope for this unit.  But in summary, the tokeniser's job is to most efficiently represent the training text in terms of tokens.  The basic steps of tokenising are:
1. Start with characters `'a'`, `'b'`, etc. as base tokens.
2. Find which sequences of tokens occur together most often (e.g. `'i'` followed by `'n'`)
3. Merge those most common sequences into new tokens.
4. Repeat until the vocabulary size limit is reached.

Let's load up the tokeniser now.  It's pulled from the oddly named online repository of AI models called [Hugging Face](https://huggingface.co/).

In [ ]:
textenc = hgf"gpt2:tokenizer"

# Vocabulary

The tokeniser effectively determines the vocabulary of the model.  In this case, the vocabulary size is $|\mathcal{V}| = 50257$, meaning there are 50,257 different tokens in GPT 2.

(The output also makes mention of the special token `unk`, standing for unknown, which resides at index 0.)

In [ ]:
@show vocab = textenc.vocab
vocab_size = length(vocab)

#
We can `lookup` tokens at a specified index.  For example, token 1234 is shown below.

In the particular BPE tokeniser used by GPT 2, the mysterious initial `Ġ` character represents a leading space.  So the token `"Ġdist"` really means `" dist"`, with the leading space.  Crucially, this is a _different token_ to `"dist"` (without the space), which in fact is token number 17081 in the vocabulary.

So in the training text, `" dist"` (with the leading space) was a commonly-occurring string, and hence was chosen to be a token in the vocabulary.  Meanwhile `"dist"` (no leading space) also occurred commonly enough to be a token, although given its index in the vocabulary it probably wasn't as common as `" dist"`.

In [ ]:
# We can look up by index
@show lookup(vocab, 1234);

# Or we can look up by token
@show lookup(vocab, "Ġdist");

# The two tokens "Ġdist" and "dist" are distinct
@show lookup(vocab, "dist");

#
The last token in the vocabulary is the special `"<|endoftext|>"` token.  If at any point during inference the model chooses to output this token, that is the signal that it has finished its output.

In [ ]:
# The last token in the vocabulary signifies end of output
end_tok = lookup(vocab, length(vocab))

#
If you really want, you can examine the whole vocabulary $\mathcal{V}$.

In [ ]:
vocab.list

# Encoding and decoding
Let's build our familiar family of encoding and decoding functions (index <--> token).  These use the functionality provided by the tokeniser to do the hard work; we're just wrapping familiar names around them.

In [ ]:
# Map index to token
idx_to_tok(i) = lookup(vocab, i)

# Map token to index
tok_to_idx(tok) = lookup(vocab, tok)

# Map string to an array of token indices
string_to_idxs(s) = copy(onecold(encode(textenc, s).token))

# Map array of token indices to a string
idxs_to_string(idxs) = join(decode(textenc, idxs))

#
Here we encode the text `"Generative AI is useful for"` into token indices.  

These indices in turn correspond to the tokens `["Gener", "ative", "ĠAI", "Ġis", "Ġuseful", "Ġfor"]`.  Notice how the leading space in some of the tokens (indicated by `Ġ`) makes it easy to compactly represent full sentences as tokens including correct spacing.


In [ ]:
# Token indices
u = string_to_idxs("Generative AI is useful for");

# The tokens themselves
@show idx_to_tok.(u)

# Now print the token indices
u

#
And here we decode the tokens back to the original text.

In [ ]:
idxs_to_string(u)

# Embedding
Just as with our CLLM, in GPT 2 the first step is to convert token indices into their vector embeddings.  In our CLLM we used an embedding dimension of just $d_e = 2$, so that the embedding could be easily visualised.  For GPT 2, the embedding dimension is $d_e = 768$.  This step works exactly the same though: there's a big embedding matrix $W_e \in \mathbb{R}^{d_e \times |\mathcal{V}|}$ and each token index just maps to its corresponding column.


# LLM architecture vs CLLM architecture

Recall our CLLM was described compactly as

$$M_p(u) = \textrm{softmax}(W_2\,\sigma.(W_1\textrm{vec}(W_e[:,u])+b_1 )+b_2)$$

where we're now using $\sigma$ to stand for a generic activation, rather than the specific $\tanh$.

That was fine for our tiny example, but it's not going to be feasible for an LLM, as we will now show. Remember we are ignoring batches in this discussion for simplicity.

In our model code we've used the symbol $X$ for the output of the embedding layer:
$$
X = W_e[:,u]\,.
$$

Introduce the notation $x^{(1)}, \ldots, x^{(C)}$ for the columns of $X$ -- that is, the separate embeddings of each token in the context.  So
$$
X = [x^{(1)}\ \cdots x^{(C)}]
$$
where $x^{(j)} \in \mathbb{R}^{d_e}$ is the vector embedding of the token index $u_j$.

Our flattening, or `vec`torisation, step stacks all these vectors into a single huge column as
$$
\textrm{vec}(X) = [x^{(1)}; \cdots ; x^{(C)}] \in \mathbb{R}^{d_eC}\,.
$$

In this notation, the dense layers that follow can be written as the function
$$
\mathcal{F}(x^{(1)}, \ldots, x^{(C)}) = W_2\, \sigma. \left(W_1[x^{(1)};  \cdots ; x^{(C)}]+b_1\right) + b_2\,.
$$

A sequence of dense layers (with intermediate activations) like this is variously referred to as a **multilayer perceptron (MLP)** or a **feed-forward network (FFN)**, both of which are terms deriving from different aspects of the history of neural nets.

The problem with these MLP layers is the number of parameters they require when the embedding size and context size are large.  Since the first weight matrix $W_1$ operates on the flattened vector $\textrm{vec}(X) \in \mathbb{R}^{d_e C}\,,$ the number of columns in the weight matrix is $d_eC$.

Our CLLM used small values for both: an embedding dimension of $d_e = 2$ and a context size of $C = 10$, that gives $\textrm{vec}(X) \in \mathbb{R}^{d_eC} = \mathbb{R}^{20}$.  You recall we picked a hidden dimension size of $16$, so that $W_1 \in \mathbb{R}^{16 \times 20}$, which was totally fine.

For an LLM such as GPT 2, even the smallest model uses an embedding dimension of $d_e = 768$, and a context size of $C = 1024$.  If we followed our CLLM model architecture, that would imply a weight matrix $W_1$ with $d_eC = 786432$ columns.  If we chose the hidden dimension to be $\mathcal{O}(d_eC)$ also (so that $W_1$ is roughly square), we'd have a weight matrix $W_1 \in \mathbb{R}^{d_eC \times d_eC} = \mathbb{R}^{786432 \times 786432}$ say.  That's more than 600 billion weights in just $W_1$ alone!


## Token-wise MLP

So it would be infeasible for an LLM to use an MLP layer that operated on the flattened embedding vectors.  Instead, they learn smaller matrices $\tilde{W}_1$ and $\tilde{W}_2$ which apply to each vector $x^{(j)}$ _separately_.  That is, we apply the _same function_
$$
\tilde{\mathcal{F}}(x^{(j)}) = \tilde{W}_2\, \sigma. (\tilde{W}_1\,x^{(j)} + \tilde{b}_1) + \tilde{b}_2
$$
to each $x^{(j)}$.

For example, if we choose a hidden dimension of $4d_e$, then $\tilde{W}_1 \in \mathbb{R}^{4d_e \times d_e} = \mathbb{R}^{3072 \times 768}$.  This is much, much better, only about $2.3$ million weights.  Similarly, $\tilde{W}_2 \in \mathbb{R}^{768 \times 3072}$ would contribute another $2.3$ million weights. And indeed, GPT 2 uses exactly this combination of two dense layers as part of its architecture.

But on its own, this **_token-wise_ MLP** can't be an adequate substitute for a proper fully connected layer across all tokens jointly.  There is no longer any way for one token to influence another!  Instead this function $\tilde{\mathcal{F}}$ provides a kind of _local_ nonlinear refinement of individual token embeddings.  Useful, yes, but it can't be the full story.  We will still need some new kind of additional layer that implements token-token interaction, or "_global_ interaction".

# Attention

The new layer that implements the token-token interaction is called **attention**, and has gained a bit of a rockstar reputation since it was introduced in the famous paper [Attention is All You Need](https://arxiv.org/pdf/1706.03762).  As a concept it predates LLMs, but its particular success as part of the **transformer** architecture that underpins modern LLMs (the 'T' in GPT) really put it on the map.

The attention layer is a true global layer
$$\mathcal{A}(x^{(1)}, \ldots, x^{(C)})$$
which allows token embeddings to influence each other. But it's also carefully crafted so that the number of learnable parameters required in the layer is under control.  Let's now go through an iterative process to design it from the ground up.

We want a layer built from matrix operations, since they can be done very efficiently on modern hardware.  But it can't be as simple as a giant fully connected layer because that would require far too many weights.  Instead, we're going to have to bake in some sort of intuition about how tokens should influence others, rather than leaving the whole mechanism up to the network to learn as a giant matrix of weights.

## Attempt 1:

For any two token embeddings $x^{(i)}$ and $x^{(j)}$, define the **attention score** $s_{ij}$ by

$$
s_{ij} = {x^{(i)}}^\top W_\textrm{score}\, x^{(j)}
$$

where $W_\textrm{score}$ is the "score matrix", to be learned.

Our plan is that this attention score will represent how much token $j$ "needs" the information that token $i$ provides.  For example, if token $j$ is `"she"` and token $i$ is `"Alice"`, then token $j$ really needs to know from token $i$ that it too, actually represents the concept `"Alice"` in a particular passage of text.

In the literature they would say that token $j$ (`"she"`) _attends to_ token $i$ (`"Alice"`), meaning that $s_{ij}$ is large.  But note that the converse need not hold, and likely wouldn't in this example.  Token $i$ (`"Alice"`) doesn't need any information from token $j$ (`"she"`) to make sense of its meaning, so $s_{ji}$ would not be large.

A word of warning: the computation above may remind you of a weighted inner product from MXB201, but it is in fact a more general **bilinear form**.  There's no reason to expect that $W_\textrm{score}$ will be symmetric (let alone positive definite) and as we've just seen we wouldn't want that.  It's good that $s_{ij} \neq s_{ji}$ in general, because they represent flows of information in opposite directions.

So the attention score $s_{ij}$ determines how much token $i$ should influence token $j$.  But what is the nature of that influence?  For that, we need a second learnable matrix $W_\textrm{contrib}$, the "contribution matrix", such that

$$
z^{(i)} = W_\textrm{contrib}\, x^{(i)}
$$

represents the (additive) contribution that comes from token $i$.  So the update to token $j$ becomes

$$
x^{(j)} \leftarrow x^{(j)} + \sum_{i=1}^C s_{ij}\, z^{(i)}\,.
$$

With such a calculation, the token for `"she"` could be strongly updated by the information flowing from the `"Alice"` token (its $z_i$), and perhaps also to a lesser extent by other tokens in the context.

## Attempt 2

There are a couple of improvements we can make to Attempt 1, even though it's certainly on the right track.

The first is that we can reduce the size of the weight matrices $W_\textrm{score} \in \mathbb{R}^{d_e \times d_e}$ and $W_\textrm{contrib} \in \mathbb{R}^{d_e \times d_e}$ by imposing low-rank factorisations
$$
W_\textrm{score} = {W_K}^\top\, W_Q \quad \textrm{and} \quad W_\textrm{contrib} = W_O W_V
$$

where
$$
W_Q, W_K, W_V \in \mathbb{R}^{d_k \times d_e},\quad W_O \in \mathbb{R}^{d_e \times d_k}
$$

for some $d_k < d_e$.

With the low-rank factorisation of $W_\textrm{score}$, the attention scores become
$$
s_{ij} = {x^{(i)}}^\top W_\textrm{score}\, x^{(j)} = {x^{(i)}}^\top {W_K}^\top\, W_Q\, x^{(j)} = ({W_K} {x^{(i)}})^\top\, (W_Q x^{(j)})
$$

and the token embedding update becomes
$$
x^{(j)} \leftarrow x^{(j)} + \sum_{i=1}^C s_{ij}\, z^{(i)} = x^{(j)} + \sum_{i=1}^C s_{ij}\, W_\textrm{contrib} x^{(i)} = x^{(j)} + \sum_{i=1}^C s_{ij}\, W_O W_V x^{(i)} = x^{(j)} + W_O \sum_{i=1}^C s_{ij}\, W_V x^{(i)}\,.
$$

We define the **query vector**
$$
q^{(j)} = {W_Q} {x^{(j)}} \in \mathbb{R}^{d_k}\,,
$$
the **key vector**
$$
k^{(i)} = W_K x^{(i)} \in \mathbb{R}^{d_k}
$$
and the **value vector**
$$
v^{(i)} = W_V x^{(i)} \in \mathbb{R}^{d_k}
$$
with the terminology reflecting their roles analogous to a kind of database retrieval problem.  With this notation, the attention scores are simply
$$
s_{ij} = k^{(i)} \cdot q^{(j)}
$$
where token $j$ is "querying" against all the "keys" stored for each token $i$ so as to retrieve and use their "values" in its update
$$
x^{(j)} \leftarrow x^{(j)} + W_O \sum_{i=1}^C s_{ij}\, v^{(i)}\,.
$$

Notice how all the query, key and value computations are ${d_k}$-dimensional calculations.  Only the final multiplication by $W_O$ projects back up to the full $d_e$-dimensional update for the token embedding.

## Attempt 3

This is an improvement in efficiency.  But we have another issue to address, which is that the scores $s_{ij}$ could be very large, very negative, and just any value at all really.  This would make the summation $\sum_i s_{ij}\, v^{(i)}$ very sensitive to small perturbations in inputs, and would likely make the training process unstable.

To solve this issue, we can call on our old friend softmax.  First we replace the raw dot products with scaled dot products
$$
s_{ij} = \frac{1}{\sqrt{d_k}} k^{(i)} \cdot q^{(j)}\,.
$$

The reasoning for the scaling is similar to the argument we used for Glorot initialisation in the last lesson.  If each component of $k^{(i)}$ and $q^{(j)}$ is roughly independent with mean zero and equal variance, then the sum of $d_k$ terms has variance roughly $d_k$ times larger.  Division by $\sqrt{d_k}$ removes that dimensional dependence.

Then we define the **attention weights**
$$
a_{ij} = \frac{\exp(s_{ij})}{\sum_k \exp(s_{kj})} = \textrm{softmax}_{\,i}(s_{ij})
$$
which, by virtue of softmax, are guaranteed to be positive and sum to one.  Softmax here is performed columnwise, as usual.

Now we have the stable update
$$
x^{(j)} \leftarrow x^{(j)} + W_O \sum_{i=1}^C a_{ij}\, v^{(i)}\,.
$$

If we perform all the computations for each embedding vector concurrently, using the matrix $X = [x^{(1)} \cdots x^{(C)}]$, then we have the formulation of attention in matrix form
$$
\begin{align*}
Q &= W_Q X \qquad\qquad\qquad\qquad\textrm{(query matrix)}\\
K &= W_K X \qquad\qquad\qquad\qquad\textrm{(key matrix)}\\
V &= W_V X \qquad\qquad\qquad\qquad\textrm{(value matrix)}\\
A &= \textrm{softmax}\left(\frac{K^\top Q}{\sqrt{d_k}}\right)\qquad\ \ \textrm{(attention matrix)}\\
X &\leftarrow X + W_O V A \qquad\qquad\quad\,\textrm{(embedding update)}
\end{align*}
$$



## Attempt 4

Attempt 3 was almost there, but there is one more subtle catch.  It turns out to make training much more efficient if we disallow "peeking ahead" in the attention mechanism.  That is, disallow tokens from later in the context to influence tokens from earlier.

This way, the model can be scored on how accurate it _would have been_ at predicting the next token after _every_ token in a training example, not just the final token.  Thus every training example comprising $C$ tokens is effectively turned into $C$ training examples, all of which are processed in parallel.  This is a massive boost in training efficiency.

However, this only works if the model is prevented from using future tokens to influence earlier ones, otherwise it would effectively be cheating during training, by having access to the very tokens it was being trained to predict.

So despite it not appearing on the t-shirts, the true attention formula used in practice is actually
$$
A = \textrm{softmax}\left(\frac{K^\top Q}{\sqrt{d_k}} + M\right)\,,
$$

where the entries of the **causal mask** $M$ are
$$
M_{ij} = \begin{cases}0, & i \leq j \\ -\infty, & i > j\end{cases}
$$

which nicely zeros out the non-causal attention weights $a_{ij}$ for $i > j$ by virtue of the calculation $\exp(-\infty) = 0$ in softmax.


# Multiheaded attention

A transformer uses **multiheaded attention**, which just means multiple attention heads ($h$ heads, say), each with their own learned weight matrices ${W_Q}_i, {W_K}_i, {W_V}_i, {W_O}_i$, each peforming an embedding update:

$$
\begin{align*}
Q_i &= {W_Q}_i X \qquad\qquad\qquad\qquad\textrm{(query matrix)}\\
K_i &= {W_K}_i X \qquad\qquad\qquad\qquad\textrm{(key matrix)}\\
V_i &= {W_V}_i X \qquad\qquad\qquad\qquad\textrm{(value matrix)}\\
A_i &= \textrm{softmax}\left(\frac{K_i^\top Q_i}{\sqrt{d_k}} + M\right)\ \textrm{(attention matrix)}\\
X &\leftarrow X + \sum_{i=1}^h {W_O}_i V_i A_i \qquad\quad\,\textrm{(embedding update)}
\end{align*}
$$

The embedding update is often presented in (equivalent) block matrix form:

$$
X \leftarrow X + [{W_O}_1 \cdots {W_O}_h] \left[\begin{array}{c}V_1 A_1 \\ \vdots \\ V_h A_h \end{array}\right] = W^O \left[\begin{array}{c}V_1 A_1 \\ \vdots \\ V_h A_h \end{array}\right]
$$
with the block output matrix $W^O = [{W_O}_1 \cdots {W_O}_h]$ interpreted as a single learnable matrix, rather than $h$ smaller learnable matrices.

Very commonly, the key dimension $d_k$ is taken to be $d_k = d_e / h$, so that $W^O \in \mathbb{R}^{d_e \times d_e}$.

Finally, we note that in the [original paper](https://arxiv.org/pdf/1706.03762), the authors assumed vectors were _rows_ rather than columns.  In that case the order of the products in many of the formulas is reversed.  

# Transformers

A transformer combines multiheaded attention with the token-wise MLP presented earlier. In our pseudo-code style notation, a single transformer block performs
$$
\begin{align*}
x^{(j)} &\leftarrow x^{(j)} + \mathcal{A}(x^{(1)}, \ldots, x^{(C)})^{(j)}\ \qquad\textrm{(multiheaded attention)}\\
x^{(j)} &\leftarrow x^{(j)} + \tilde{\mathcal{F}}(x^{(j)})\quad\qquad\qquad\qquad\textrm{(token-wise MLP)}
\end{align*}
$$

or in purely mathematical notation,
$$
{x^{(j)}}^{(\textrm{new})} = x^{(j)} + \mathcal{A}(x^{(1)}, \ldots, x^{(C)})^{(j)} + \tilde{\mathcal{F}}\left(x^{(j)}+ \mathcal{A}(x^{(1)}, \ldots, x^{(C)})^{(j)}\right)\,.
$$

Notice the very deliberate [residual](https://en.wikipedia.org/wiki/Residual_neural_network) style, where $x^{(j)}$ is _updated_ from its current value with a correction, rather than being replaced, i.e. $x^{(j)} \leftarrow x^{(j)} + f(x^{(j)})$ rather than $x^{(j)} \leftarrow f(x^{(j)})$.  We didn't need this trick for our earlier CLLM, but for deep transformer-based networks it's essential.  This way each layer needs to only learn the _correction_ required to the previous layer, which greatly improves the stability and efficiency of the training process.

The picture of a deep transformer network is really one of tokens being progressively _refined_ from their initial projections in embedding space, as they are influenced by more and more information originating from other tokens.  After passing through a sequence of transfomer blocks, the representation of each token in embedding space should ideally capture all of the semantic information there is to be gleaned from the surrounding context.  So `"she"` is the girl Alice, living in the Victorian era, who tends to speak in a certain way, and so on.

There are also some additional components to a transformer which are related to stabilising the training process (dropout and normalisation layers which we won't get into.

# GPT 2

A full GPT 2 model comprises the embedding layer, a sequence of transformer blocks, and a final layer to output the predicted probabilities.  Here are the details of the smallest GPT 2 model, which is the model we will be using.

| Parameter | Symbol | Value |
|---|---|---|
| Vocabulary Size | $\vert\mathcal{V}\vert$ | 50,257 |
| Context size | $C$ | 1024 |
| Batch size | $B$ | 512 |
| Embedding dimension | $d_e$ | 768 |
| Key dimension | $d_k$ | 64 |
| Attention heads per block | $h$ | 12 |
| Transformer blocks |  | 12 |


And here it is in all its glory.  Notice we're definitely not building (let alone training!) this from scratch.  We're loading a pre-built, pre-trained model from the Hugging Face repository, in which all the machinery for the layers is bundled up inside.  Altogether, there are 124,439,808 trainable parameters in the model, but they too are buried inside this opaque `HGFGPT2LMHeadModel` data type.  The intended use of this pre-trained model is to perform inference, not to go poking around in its internals.

Having said that, if you stare hard enough at the output you can see that all the layers we spoke about above are indeed present and correct, with the dimensions as given (e.g. the `Embed` layer right near the top has a size of $768 \times 50257$, which is just right for projecting any of the 50,257 possible tokens onto its 768-dimensional embedding).

In [ ]:
gpt2arch = hgf"gpt2:lmheadmodel"

#
If you really, really want to pull out the parameters and take a look you can, but it's awkward.  Here's the embedding matrix for example.

In [ ]:
We = gpt2arch.model.embed.layer.token.embeddings

# Positional embedding

There is one final point to make about the token representation in the model.  Thus far we have presented tokens as being initially represented by their projections into the embedding space.  Actually that's not quite true.  The real initial representation of a token is this embedding, _plus_ a so-called **positional embedding**.  You can see the blurb about this `FixedLenPositionEmbed` in the output above.

It's not a big change.  Along with the ordinary embedding matrix $W_e \in \mathbb{R}^{d_e \times |\mathcal{V}|}$, we also now have a _positional_ embedding matrix $E \in \mathbb{R}^{d_e \times C}$.


In [ ]:
E = gpt2arch.model.embed.layer.position.embed.embeddings

#
The initial token embeddings are the _sum_ of their respective columns in $W_e$ and $E$.  The relevant column of $W_e$ as we know is determined by the token index.  Whereas the relevant column of $E$ is simply the token's position in the context.  For example, the first token in our string `"Generative AI is good for"` has index $u_1 = 8646$.  So its initial embedding is the sum $x_1 = W_e[:, u_1] + E[:, 1] = W_e[:, 8646] + E[:, 1]$.  The second token has index $u_2 = 877$ so its initial embedding is the sum $x_2 = W_e[:, u_2] + E[:, 2] = W_e[:, 877] + E[:, 2]$.

Both matrices $W_e$ and $E$ are of course learned during training.  So the model is free to choose these weights in whatever way helps it infer the most meaning from the text (e.g. by using the positional embedding to notice which adjectives come _before_ a particular noun, or whatever).

It's worth emphasising that throughout the entire model, $E$ is the _only_ matrix of weights that depends on the context size.  None of the MLP layer matrices do (because they all act per-token only) and none of the attention layer matrices do (because the interaction arises through query-key vector dot products and value vector updates).  So the context size of a model does not strongly influence the number of parameters in the model.  Context sizes are instead limited by the memory and runtime cost considerations of the attention mechanism.

In [ ]:
# This would be the actual representation of the first token entering the transformer layers
x1 = We[:, u[1]] + E[:,1]

# Inference

It might be big and fancy (by our standards!), but the GPT 2 model is still just a function that takes in an array of token indices, and outputs an array of probabilities:
$$
\hat{Y} = M_p(u)\,.
$$

(Actually the model only outputs logits, and it's up to us to turn those into probabilities.)  Let's give it a try on our little sentence `"Generative AI is useful for"` from earlier.

The data structure representing the network has its own convention for how to pass the input $u$. And as already noted, the parameters $p$ are actually buried within the data structure itself, so we don't pass them in separately.

We see the output has one column for each token in the input: column $j$ contains the logits for predicting the next token following the first $j$ tokens in the input sequence.

In [ ]:
# This is the format the input layer expects
input = (; token=u)

# And the output logits
output = gpt2arch(input).logit

#
Here are those predictions using `argmax` sampling.

We can compare the next token predictions against what actually came next in the text.  It actually predicted two of them correctly!  After `"Generative AI"` it correctly predicted `" is"`; and after `"Generative AI is useful"` it correctly predicted `" for"`.

Its prediction for the next token following the full input text is `" many"`, which also seems fair enough.

In [ ]:
predictions = argmax.(eachcol(output[:,:]))
@show predictions

for j = 1:length(u)
  println(idxs_to_string([u[1:j]; predictions[j]]))
end

#
For inference (as opposed to training) we really only want the final column of the output. So let's build our familiar `model` function.  We just need to forward the input vector on to `gpt2arch` in the format it expects and run a `softmax` on the final column of the output.




In [ ]:
function model(u)

  # input is just our token indices
  input = (; token=u)

  # run it through the model
  output = gpt2arch(input)

  # gpt2arch doesn't include the final softmax layer, you're expected to do that yourself
  return softmax(output.logit[:, end])

end

#
Running the `model` on our little sentence `"Generative AI is useful for"` we get the probabilities it has determined for the next token prediction.

In [ ]:
 ŷ = model(u)

#
From here we could sample to generate statistically plausible continuations.  Or we could just take the `argmax` to find the most likely next token, which we saw earlier is `" many"`.

In [ ]:
@show idx_to_tok(argmax(ŷ))
idxs_to_string([u; argmax(ŷ)])

# Generating new text

Just like with our character level language model from the previous lessons, to generate new text we run the model on the initial context, append one new token, and repeat.

There is one difference though: for our CLLM we had to manually "slide" the context window to encompass the 10 most recent characters.  The `gpt2arch` object does that for us, so we can input a context of whatever length we like, and (at most) the 1024 most recent tokens will actually be used for inference.

In [ ]:
function generate_text(context, max_new_tokens=10; interactive=false, break_idxs=[])

    # Tokenise the context
    idxs = string_to_idxs(context)
    # Generate new tokens one by one
    for i = 1:max_new_tokens
        probs = model(idxs)
        new_id = sample(1:vocab_size, Weights(probs))
        if new_id in [break_idxs; vocab_size]
          break # generating <|endoftext|> means we're done early
        else
          push!(idxs, new_id) # append the new token id
          if interactive # chatbot style
            print(idxs_to_string(new_id))
          end
        end
    end

    # Convert the token indices to text
    return idxs_to_string(idxs)
end

#
Let's try it out on our favourite sentence a few times.  We'll generate 20 new tokens each time.

It sure spouts a lot of nonsense.  But now you fully understand why.  All it's actually doing is repeatedly predicting the next token to produce new text that is statistically consistent with the text it was trained on.  There's certainly no magic here.

In [ ]:
for reps = 1:5
  println(generate_text("Generative AI is useful for", 20))
end

# Chatbots

There is more work required to turn a next token predictor like GPT into a chatbot.  Mostly this is out of scope of the unit, but you can read about things like [reinforcement learning from human feedback](https://en.wikipedia.org/wiki/Rlhf) (RLHF) online.

Without any further work we can build a very rough attempt all the same.  The idea is to seed the model with a context that looks like a transcript of a conversation between a human and a helpful AI assistant.  By continuing the conversation from there with next token prediction, the model will therefore effectively be assuming the role of an actual helpful AI assistant.

In [ ]:
function chatbot(max_new_tokens=50)

  context = """
The following is the transcript of a conversation between a human and a helpful AI assistant.

## Human: Hi.
## AI: Hello, I'm a helpful AI assistant. How can I help you?

## Human: I'd like to ask you some questions, is that OK?
## AI: Yes, I would be delighted to answer your questions accurately and succinctly.

## Human: What's the capital of France?
## AI: The capital of France is Paris.

## Human: Who developed the theory of gravity?
## AI: Isaac Newton developed the first theory of gravity. Albert Einstein modified it to include relativity.

"""

  while true
    prompt = readline()
    if isempty(prompt) break end
    context *= "## Human: " * prompt * "\n## AI:"
    context = generate_text(context, max_new_tokens; interactive=true, break_idxs=tok_to_idx.("##"))
  end

  return context
end

#
You can give it a try below if you're brave enough!  Be warned though: it will very definitely say all kinds of unfiltered nonsense.

In [ ]:
Random.seed!(0)  # in case you care about reproducing the nonsense

# This will prompt you to enter some text to start (continue, really) the conversation.
# Enter a blank line to finish.
transcript = chatbot();

#
The context is always being appended to, so by the end of the conversation it contains the full transcript of what was said.

In [ ]:
println(transcript)

# Next token prediction: good enough?  Yes!

You might be wondering, is next token prediction really sufficient to generate statistically plausible new text at scale?  Mightn't it be necessary to predict several tokens at a time, or a whole sentence, or a paragraph, in order that the generated text really looks like it was drawn from the same distribution as the training data?

In fact, next token prediction is good enough.

Let $P(v_1, v_2\ |\ c)$ denote the joint probability distribution of the next two predicted tokens $v_1$ and $v_2$, given the context $c$.  An autoregressive model, such as an LLM, does not model this joint probability directly.  But from the [chain rule of probability](https://en.wikipedia.org/wiki/Chain_rule_(probability),

$$
P(v_1, v_2\ |\ c) = P(v_1\ |\ c)\ P(v_2\ |\ v_1, c)\,.
$$

So drawing a token $v_1$ given the context $c$, _and then appending that new token to the context_ before drawing a second token $v_2$ is mathematically equivalent to drawing a pair of tokens $(v_1, v_2)$ given the context $c$.

The same is true for any number of tokens, so we have the general **autoregressive factorisation**

$$
P(v_1, \ldots, v_K\ |\ c) = \prod_{k = 1}^K P(v_k\ |\ v_1, \ldots, v_{k-1}, c)\,.
$$

So next-token prediction with appending is really just a super-efficient method for drawing a sample from the full joint probability distribution.  This explains why even the most modern LLMs do not try to predict any more than one token at a time: they don't need to.  All of the innovation that goes into improving LLMs can be focused on just next token prediction.

# What changes in more modern LLMs?

Every little aspect of GPT 2 has been improved upon over the years, but the overall picture of embedding -> transformers -> softmax remains the backbone of modern LLMs.  Your favourite chatbot is still fundamentally an autoregressive transformer trained on next token prediction.

Mostly it's about scale.  In fact, if all you did was to:

* dramatically increase the number and size of the transformer layers in GPT 2,
* so that the parameter count was a few hundred billion,
* and trained it with a huge curated data set comprising trillions of tokens,
* for several months,
* using tens of thousands of GPUs,
* at a cost of several hundred million dollars,
* and several GWh of electricity

you'd probably have something that performs comparably to a modern day LLM.

Speaking of training, it is a substantial engineering feat to train very large models.  Aside from the enormous computational cost which already puts it out of reach of most organisations, let along individuals, there are also issues of data communication, optimiser stability, fault tolerance, hyperparameter tuning, and numerous other concerns.  But these issues are outside the scope of this unit.

# Conclusion

In this lesson we learned:

* about the process of tokenising
* how tokens are represented as learned embedding vectors together with positional information
* how the attention mechanism allows tokens to exchange information
* how query, key and value vectors are used to compute attention weights between tokens
* how causal masking prevents a token from accessing future tokens during training
* how transformers are built from attention layers, MLP layers and residual connections
* how next-token prediction can generate coherent text one token at a time

This concludes our exploration into AI language models.